# **Fraud Detection — End to End ML Project**

**Dataset:** Synthetic Financial Datasets (PaySim) — 6.3M transactions  
**Model:** XGBoost Classifier (V2 with engineered features)  
**API:** Flask REST API with authentication, deployed via Docker  

---

## **Table of Contents**
1. Setup & Data Loading
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Preprocessing & Class Imbalance
5. Model Training & Comparison
6. Threshold Tuning
7. Final Evaluation
8. API Test

## **1. Setup & Data Loading**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay, precision_recall_curve
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import joblib

plt.style.use('dark_background')
COLORS = {
    'fraud':   '#ff4757',
    'legit':   '#00ff88',
    'neutral': '#6b7280',
    'accent':  '#3b82f6'
}

print('Libraries loaded ✓')

In [ ]:
df = pd.read_csv('../data/Synthetic_Financial_datasets_log.csv')
print(f'Shape      : {df.shape}')
print(f'Columns    : {list(df.columns)}')
print(f'Fraud cases: {df["isFraud"].sum():,} ({df["isFraud"].mean()*100:.3f}%)')
df.head()

In [ ]:
print(df.info())
print('\nMissing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

## **2. Exploratory Data Analysis**

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['isFraud'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values,
            color=[COLORS['legit'], COLORS['fraud']], alpha=0.85, width=0.5)
axes[0].set_title('Transaction class distribution', color='white')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5000, f'{v:,}', ha='center', color='white', fontsize=11)

# Fraud by transaction type
type_fraud = df.groupby('type')['isFraud'].mean() * 100
type_fraud.plot(kind='bar', ax=axes[1], color=COLORS['fraud'], alpha=0.85, rot=0)
axes[1].set_title('Fraud rate by transaction type (%)', color='white')
axes[1].set_ylabel('Fraud rate %')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

print('\nFraud only occurs in TRANSFER and CASH_OUT transactions')

In [ ]:
# Amount distribution — fraud vs legit
fraud = df[df['isFraud'] == 1]
legit = df[df['isFraud'] == 0].sample(5000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(np.log1p(legit['amount']), bins=50, color=COLORS['legit'], alpha=0.6, label='Legit', density=True)
axes[0].hist(np.log1p(fraud['amount']), bins=50, color=COLORS['fraud'], alpha=0.6, label='Fraud', density=True)
axes[0].set_title('Amount distribution (log scale)', color='white')
axes[0].set_xlabel('log(amount + 1)')
axes[0].legend()

# Balance patterns
axes[1].scatter(legit['oldbalanceOrg'].sample(1000), legit['newbalanceOrig'].sample(1000),
                alpha=0.3, s=8, color=COLORS['legit'], label='Legit')
axes[1].scatter(fraud['oldbalanceOrg'], fraud['newbalanceOrig'],
                alpha=0.5, s=8, color=COLORS['fraud'], label='Fraud')
axes[1].set_title('Sender: balance before vs after', color='white')
axes[1].set_xlabel('oldbalanceOrg')
axes[1].set_ylabel('newbalanceOrig')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Outlier analysis
num_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

fig, axes = plt.subplots(1, len(num_cols), figsize=(16, 4))
for ax, col in zip(axes, num_cols):
    ax.boxplot(df[col].clip(upper=df[col].quantile(0.99)), patch_artist=True,
               boxprops=dict(facecolor=COLORS['accent'], alpha=0.5),
               medianprops=dict(color=COLORS['legit']))
    ax.set_title(col, color='white', fontsize=10)
    ax.tick_params(colors=COLORS['neutral'])

fig.suptitle('Outlier analysis — numerical features (clipped at 99th percentile)', color='white')
plt.tight_layout()
plt.show()

print('Outliers are kept — extreme values are the fraud signal we want to learn')

## **3. Feature Engineering**

Two key derived features encode the fraud pattern directly:

- `balance_error_orig` = `oldbalanceOrg - newbalanceOrig - amount`  
  Should be 0 for honest transactions. Non-zero = money anomaly on sender side.

- `balance_error_dest` = `newbalanceDest - oldbalanceDest - amount`  
  Should be 0 normally. Strongly negative when money vanishes (layering pattern).

In [ ]:
# Clean and encode
df = df.dropna()
df = df.drop(columns=['nameOrig', 'nameDest'])
df['type'] = df['type'].astype('category').cat.codes

# Engineer features
df['balance_error_orig'] = df['oldbalanceOrg'] - df['newbalanceOrig'] - df['amount']
df['balance_error_dest'] = df['newbalanceDest'] - df['oldbalanceDest'] - df['amount']

fraud = df[df['isFraud'] == 1]
legit = df[df['isFraud'] == 0].sample(5000, random_state=42)

print('Balance error stats:')
print(f'  balance_error_orig — fraud mean: {fraud["balance_error_orig"].mean():>15,.2f}')
print(f'  balance_error_orig — legit mean: {legit["balance_error_orig"].mean():>15,.2f}')
print(f'  balance_error_dest — fraud mean: {fraud["balance_error_dest"].mean():>15,.2f}')
print(f'  balance_error_dest — legit mean: {legit["balance_error_dest"].mean():>15,.2f}')

In [ ]:
# Visualize the separation power of engineered features
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Engineered features — fraud vs legitimate separation', color='white')

for ax, col, title in zip(axes,
    ['balance_error_orig', 'balance_error_dest'],
    ['Origin balance error', 'Destination balance error']):
    ax.hist(np.clip(legit[col], -5e5, 5e5), bins=60, alpha=0.6,
            color=COLORS['legit'], label='Legit', density=True)
    ax.hist(np.clip(fraud[col], -5e5, 5e5), bins=60, alpha=0.6,
            color=COLORS['fraud'], label='Fraud', density=True)
    ax.set_title(title, color='white')
    ax.set_xlabel('Error value (clipped)')
    ax.legend()

plt.tight_layout()
plt.show()

## **4. Preprocessing & Class Imbalance**

In [ ]:
FEATURES_V1 = ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
                'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud']
FEATURES_V2 = FEATURES_V1 + ['balance_error_orig', 'balance_error_dest']
NUM_COLS    = ['amount', 'oldbalanceOrg', 'oldbalanceDest', 'step',
               'balance_error_orig', 'balance_error_dest']

X = df[FEATURES_V2]
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train size : {len(X_train):,}')
print(f'Test size  : {len(X_test):,}')
print(f'\nClass distribution (train):')
print(y_train.value_counts(normalize=True).round(4))

In [ ]:
# Scale
scaler = StandardScaler()
X_train_sc = X_train.copy()
X_test_sc  = X_test.copy()
X_train_sc[NUM_COLS] = scaler.fit_transform(X_train[NUM_COLS])
X_test_sc[NUM_COLS]  = scaler.transform(X_test[NUM_COLS])

# SMOTE
smote = SMOTE(sampling_strategy=0.2, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)

print('After SMOTE:')
print(pd.Series(y_train_sm).value_counts())

# Visualize before/after
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, counts, title in zip(axes,
    [y_train.value_counts(), pd.Series(y_train_sm).value_counts()],
    ['Before SMOTE', 'After SMOTE']):
    ax.bar(['Legit', 'Fraud'], counts.values,
           color=[COLORS['legit'], COLORS['fraud']], alpha=0.8, width=0.5)
    ax.set_title(title, color='white')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## **5. Model Training & Comparison**

### **Key bug fixed from original notebook**
The original notebook called `cross_val_score()` then saved the model with `joblib.dump()`.  
`cross_val_score` fits temporary internal clones — it does **not** fit the model object itself.  
The saved model was an unfitted shell, which is why every prediction returned the error:  
`need to call fit or load_model beforehand`

**Fix:** always call `model.fit()` explicitly after cross-validation.

In [ ]:
# V1 — original features only
FEATURES_V1_SC = FEATURES_V1  # V1 uses same columns but no engineered features
NUM_COLS_V1 = ['amount', 'oldbalanceOrg', 'oldbalanceDest', 'step']

scaler_v1 = StandardScaler()
X_train_v1 = X_train[FEATURES_V1].copy()
X_test_v1  = X_test[FEATURES_V1].copy()
X_train_v1[NUM_COLS_V1] = scaler_v1.fit_transform(X_train_v1[NUM_COLS_V1])
X_test_v1[NUM_COLS_V1]  = scaler_v1.transform(X_test_v1[NUM_COLS_V1])

X_train_sm_v1, y_train_sm_v1 = SMOTE(sampling_strategy=0.2, random_state=42).fit_resample(X_train_v1, y_train)

model_v1 = XGBClassifier(
    subsample=1.0, reg_lambda=1, reg_alpha=0.1,
    n_estimators=300, max_depth=7, learning_rate=0.2,
    gamma=0.5, colsample_bytree=0.8, random_state=42, eval_metric='logloss'
)
model_v1.fit(X_train_sm_v1, y_train_sm_v1)
print('V1 trained ✓')

# V2 — with engineered features
model_v2 = XGBClassifier(
    subsample=1.0, reg_lambda=1, reg_alpha=0.1,
    n_estimators=300, max_depth=7, learning_rate=0.2,
    gamma=0.5, colsample_bytree=0.8, random_state=42, eval_metric='logloss'
)
model_v2.fit(X_train_sm, y_train_sm)
print('V2 trained ✓')

In [ ]:
# Evaluate both
THRESHOLD = 0.6835

y_prob_v1 = model_v1.predict_proba(X_test_v1)[:, 1]
y_prob_v2 = model_v2.predict_proba(X_test_sc)[:, 1]
y_pred_v1 = (y_prob_v1 >= THRESHOLD).astype(int)
y_pred_v2 = (y_prob_v2 >= THRESHOLD).astype(int)

auc_v1 = roc_auc_score(y_test, y_prob_v1)
auc_v2 = roc_auc_score(y_test, y_prob_v2)
rep_v1 = classification_report(y_test, y_pred_v1, output_dict=True)
rep_v2 = classification_report(y_test, y_pred_v2, output_dict=True)

print(f'  {"Metric":<25} {"V1 (baseline)":>14} {"V2 (engineered)":>16} {"Change":>10}')
print(f'  {"-"*68}')
print(f'  {"ROC-AUC":<25} {auc_v1:>14.4f} {auc_v2:>16.4f} {auc_v2-auc_v1:>+10.4f}')
for metric in ['precision', 'recall', 'f1-score']:
    v1 = rep_v1['1'][metric]
    v2 = rep_v2['1'][metric]
    print(f'  {"Fraud "+metric:<25} {v1:>14.4f} {v2:>16.4f} {v2-v1:>+10.4f}')

In [ ]:
# Full comparison plots
fig = plt.figure(figsize=(16, 10))
fig.suptitle('Model Comparison — V1 (baseline) vs V2 (engineered features)', fontsize=13, color='white')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ROC
ax1 = fig.add_subplot(gs[0, 0])
RocCurveDisplay.from_predictions(y_test, y_prob_v1, ax=ax1, name=f'V1 AUC={auc_v1:.4f}', color=COLORS['neutral'])
RocCurveDisplay.from_predictions(y_test, y_prob_v2, ax=ax1, name=f'V2 AUC={auc_v2:.4f}', color=COLORS['accent'])
ax1.set_title('ROC Curve', color='white')

# PR curve
ax2 = fig.add_subplot(gs[0, 1])
PrecisionRecallDisplay.from_predictions(y_test, y_prob_v1, ax=ax2, name='V1', color=COLORS['neutral'])
PrecisionRecallDisplay.from_predictions(y_test, y_prob_v2, ax=ax2, name='V2', color=COLORS['accent'])
ax2.set_title('Precision-Recall Curve', color='white')

# Bar chart
ax3 = fig.add_subplot(gs[0, 2])
metrics = ['Precision', 'Recall', 'F1']
v1_vals = [rep_v1['1']['precision'], rep_v1['1']['recall'], rep_v1['1']['f1-score']]
v2_vals = [rep_v2['1']['precision'], rep_v2['1']['recall'], rep_v2['1']['f1-score']]
x = np.arange(len(metrics))
ax3.bar(x - 0.2, v1_vals, 0.35, label='V1', color=COLORS['neutral'], alpha=0.8)
ax3.bar(x + 0.2, v2_vals, 0.35, label='V2', color=COLORS['accent'], alpha=0.8)
ax3.set_xticks(x); ax3.set_xticklabels(metrics)
ax3.set_ylim(0, 1.1); ax3.set_title('Fraud class metrics', color='white')
ax3.legend()

# Confusion matrices
for ax, y_pred, title in zip(
    [fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])],
    [y_pred_v1, y_pred_v2],
    ['Confusion Matrix — V1', 'Confusion Matrix — V2']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt=',', cmap='RdYlGn', ax=ax,
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    ax.set_title(title, color='white')

# Feature importance
ax5 = fig.add_subplot(gs[1, 2])
importance = pd.Series(model_v2.feature_importances_, index=FEATURES_V2).sort_values()
bar_colors = [COLORS['accent'] if 'error' in f else COLORS['neutral'] for f in importance.index]
importance.plot(kind='barh', ax=ax5, color=bar_colors)
ax5.set_title('Feature importance — V2\n(blue = engineered)', color='white')

plt.show()

## **6. Threshold Tuning**

The default 0.5 threshold is rarely optimal for imbalanced fraud data.  
We use the precision-recall curve to find the threshold that maximises F1.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob_v2)
f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
best_idx = f1.argmax()
best_threshold = thresholds[best_idx]

print(f'Optimal threshold : {best_threshold:.4f}')
print(f'At optimal        → Precision: {precision[best_idx]:.4f}  Recall: {recall[best_idx]:.4f}  F1: {f1[best_idx]:.4f}')

# Plot threshold vs precision/recall/f1
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, precision[:-1], color=COLORS['accent'],  label='Precision', linewidth=1.5)
ax.plot(thresholds, recall[:-1],    color=COLORS['legit'],   label='Recall',    linewidth=1.5)
ax.plot(thresholds, f1[:-1],        color=COLORS['fraud'],   label='F1',        linewidth=1.5)
ax.axvline(best_threshold, color='white', linestyle='--', alpha=0.6, label=f'Optimal ({best_threshold:.4f})')
ax.axvline(0.5,            color=COLORS['neutral'], linestyle=':', alpha=0.6, label='Default (0.5)')
ax.set_xlabel('Threshold')
ax.set_title('Precision / Recall / F1 vs decision threshold', color='white')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

## **7. Final Evaluation**

In [ ]:
y_pred_final = (y_prob_v2 >= best_threshold).astype(int)

print('=== Final Model (V2) — Test Set Evaluation ===')
print(f'ROC-AUC Score : {roc_auc_score(y_test, y_prob_v2):.4f}')
print(f'Threshold     : {best_threshold:.4f}')
print()
print(classification_report(y_test, y_pred_final, target_names=['Legitimate', 'Fraud']))

cm = confusion_matrix(y_test, y_pred_final)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt=',', cmap='RdYlGn', ax=ax,
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
ax.set_title('Final confusion matrix', color='white')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True positives  (fraud caught)     : {tp:,}')
print(f'False negatives (fraud missed)     : {fn:,}')
print(f'False positives (legit flagged)    : {fp:,}')
print(f'True negatives  (legit cleared)    : {tn:,}')

## **8. API Test**

Test the running Flask API with known fraud and legitimate scenarios.  
Make sure `python app.py` is running before executing this cell.

In [ ]:
import requests
from dotenv import load_dotenv
import os

load_dotenv('../.env')
API_KEY = os.getenv('API_KEY')

URL     = 'http://127.0.0.1:5000/predict'
HEADERS = {'X-API-Key': API_KEY}

scenarios = [
    {
        'label': 'Legitimate PAYMENT',
        'payload': {'step':1,'type':3,'amount':4878.0,'oldbalanceOrg':170136.0,
                    'newbalanceOrig':165258.0,'oldbalanceDest':0.0,'newbalanceDest':4878.0,'isFlaggedFraud':0}
    },
    {
        'label': 'Fraudulent TRANSFER (account drain + layering)',
        'payload': {'step':1,'type':4,'amount':450000.0,'oldbalanceOrg':450000.0,
                    'newbalanceOrig':0.0,'oldbalanceDest':0.0,'newbalanceDest':0.0,'isFlaggedFraud':0}
    },
    {
        'label': 'Fraudulent CASH_OUT (full drain)',
        'payload': {'step':1,'type':1,'amount':200000.0,'oldbalanceOrg':200000.0,
                    'newbalanceOrig':0.0,'oldbalanceDest':0.0,'newbalanceDest':0.0,'isFlaggedFraud':0}
    }
]

for s in scenarios:
    r = requests.post(URL, json=s['payload'], headers=HEADERS).json()
    emoji = '🚨' if r['prediction'] == 1 else '✅'
    print(f"{emoji} {s['label']}")
    print(f"   label={r['label']}  probability={r['fraud_probability']}  threshold={r['threshold_used']}")
    print()